In [0]:
# Working on the data quality frameworks

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# --------------------------------------------------
# 1. Reusable validation functions
# --------------------------------------------------

quality_results = []

def check_records(check_name, table_name, failed_records):
    result = "PASS" if failed_records == 0 else "FAIL"

    quality_results.append((
        check_name,
        table_name,
        int(failed_records),
        result
    ))

    print(f"{check_name}: {result} ({failed_records} failed records)")


def check_duplicates(df, key_columns):
    return (
        df.groupBy(key_columns)
        .count()
        .filter(
            (F.col("count") > 1) &
            F.expr(
                " AND ".join(
                    [f"`{column}` IS NOT NULL" for column in key_columns]
                )
            )
        )
        .count()
    )


# --------------------------------------------------
# 2. BRONZE - Appointment checks
# --------------------------------------------------

appointments = spark.table(
    "healthcare_demo.bronze.appointments"
)

check_records(
    "Duplicate appointment IDs",
    "bronze.appointments",
    check_duplicates(appointments, ["appointment_id"])
)

check_records(
    "Missing appointment required fields",
    "bronze.appointments",
    appointments.filter(
        F.col("appointment_id").isNull() |
        F.col("referral_id").isNull() |
        F.col("appointment_date").isNull() |
        F.col("location_id").isNull()
    ).count()
)

check_records(
    "Invalid appointment statuses",
    "bronze.appointments",
    appointments.filter(
        ~F.col("appointment_status").isin(
            "Booked", "Attended", "Cancelled"
        ) |
        F.col("appointment_status").isNull()
    ).count()
)


# --------------------------------------------------
# 3. BRONZE - Capacity checks
# --------------------------------------------------

capacity = spark.table(
    "healthcare_demo.bronze.capacity"
)

check_records(
    "Duplicate clinic specialty combinations",
    "bronze.capacity",
    check_duplicates(capacity, ["location_id", "specialty"])
)

check_records(
    "Missing capacity information",
    "bronze.capacity",
    capacity.filter(
        F.col("location_id").isNull() |
        F.col("location_name").isNull() |
        F.col("specialty").isNull() |
        F.col("available_slots").isNull()
    ).count()
)

check_records(
    "Negative available slots",
    "bronze.capacity",
    capacity.filter(
        F.col("available_slots") < 0
    ).count()
)


# --------------------------------------------------
# 4. SILVER - Patient referral checks
# --------------------------------------------------

silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

check_records(
    "Duplicate referral IDs",
    "silver.patient_referral_details",
    check_duplicates(silver, ["referral_id"])
)

check_records(
    "Missing mandatory patient referral fields",
    "silver.patient_referral_details",
    silver.filter(
        F.col("patient_id").isNull() |
        F.col("referral_id").isNull() |
        F.col("specialty").isNull() |
        F.col("priority").isNull() |
        F.col("referral_date").isNull()
    ).count()
)

check_records(
    "Invalid priority values",
    "silver.patient_referral_details",
    silver.filter(
        ~F.col("priority").isin("Urgent", "Routine") |
        F.col("priority").isNull()
    ).count()
)

check_records(
    "Invalid specialty values",
    "silver.patient_referral_details",
    silver.filter(
        ~F.col("specialty").isin(
            "Cardiology", "Dermatology", "Orthopaedics"
        ) |
        F.col("specialty").isNull()
    ).count()
)

check_records(
    "Negative waiting days",
    "silver.patient_referral_details",
    silver.filter(
        F.col("waiting_days") < 0
    ).count()
)

check_records(
    "Missing waiting days",
    "silver.patient_referral_details",
    silver.filter(
        F.col("waiting_days").isNull()
    ).count()
)

check_records(
    "Appointment before referral date",
    "silver.patient_referral_details",
    silver.filter(
        F.col("appointment_date").isNotNull() &
        F.col("referral_date").isNotNull() &
        (F.col("appointment_date") < F.col("referral_date"))
    ).count()
)


# --------------------------------------------------
# 5. GOLD - Patient allocation checks
# --------------------------------------------------

recommendations = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

check_records(
    "Duplicate patient referral allocations",
    "gold.appointment_recommendations",
    check_duplicates(recommendations, ["referral_id"])
)

check_records(
    "Recommended patients missing clinic",
    "gold.appointment_recommendations",
    recommendations.filter(
        (F.col("allocation_status") == "Recommended") &
        (
            F.col("recommended_location_id").isNull() |
            F.col("recommended_location_name").isNull()
        )
    ).count()
)

# Match recommended patients to confirmed clinic capacity
recommended = recommendations.filter(
    F.col("allocation_status") == "Recommended"
)

capacity_for_check = capacity.select(
    F.col("location_id").alias("capacity_location_id"),
    F.col("specialty").alias("capacity_specialty"),
    F.col("available_slots").alias("capacity_slots")
)

capacity_match = recommended.join(
    capacity_for_check,
    (
        (recommended["recommended_location_id"] ==
         capacity_for_check["capacity_location_id"]) &
        (recommended["specialty"] ==
         capacity_for_check["capacity_specialty"])
    ),
    "left"
)

check_records(
    "Recommended patients without confirmed capacity",
    "gold.appointment_recommendations",
    capacity_match.filter(
        F.col("capacity_slots").isNull()
    ).count()
)

capacity_counts = (
    capacity_match
    .groupBy(
        "recommended_location_id",
        "specialty"
    )
    .agg(
        F.countDistinct("referral_id").alias("recommended_count"),
        F.max("capacity_slots").alias("available_slots")
    )
)

check_records(
    "Clinic capacity exceeded",
    "gold.appointment_recommendations",
    capacity_counts.filter(
        F.col("recommended_count") > F.col("available_slots")
    ).count()
)


# --------------------------------------------------
# 6. Create data quality results DataFrame
# --------------------------------------------------

result_schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("failed_records", IntegerType(), False),
    StructField("result", StringType(), False)
])

quality_df = spark.createDataFrame(
    quality_results,
    result_schema
).withColumn(
    "checked_at",
    F.current_timestamp()
)


# --------------------------------------------------
# 7. Save results into Gold audit table
# --------------------------------------------------

quality_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("healthcare_demo.gold.data_quality_audit")


# --------------------------------------------------
# 8. Display results
# --------------------------------------------------

display(
    quality_df.orderBy("source_table", "check_name")
)

failed_checks = quality_df.filter(
    F.col("result") == "FAIL"
).count()

if failed_checks == 0:
    print("All data quality checks PASSED")
else:
    print(f"{failed_checks} data quality checks FAILED")


In [0]:
display(
    spark.table("healthcare_demo.gold.data_quality_audit")
)

In [0]:
quality_df.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_demo.gold.data_quality_results"
)

In [0]:
display(
    spark.table("healthcare_demo.gold.data_quality_results")
    .orderBy("source_table", "check_name")
)